# Mistral 7B v0.3 PEFT Tuning on Sentiment140

Parameter-efficient fine-tuning (PEFT) on Mistral-7B-v0.3 for binary sentiment classification using the Sentiment140 dataset. We compare LoRA and Adapter methods, optimized via Optuna.


In [ ]:
!pip install -q transformers peft bitsandbytes datasets optuna evaluate scikit-learn accelerate tqdm pandas adapter-transformers huggingface_hub


In [ ]:
import torch
import pandas as pd
import numpy as np
import optuna
import evaluate
from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from peft import LoraConfig, IA3Config, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig
from huggingface_hub import HfApi

def gpu_memory_gb():
    """Return peak GPU memory allocated in GB (resets the counter)."""
    peak = torch.cuda.max_memory_allocated() / 1024**3
    torch.cuda.reset_peak_memory_stats()
    return round(peak, 3)


## Dataset
Load and preprocess the Sentiment140 dataset (5k train, 1k test).


In [ ]:
def prepare_dataset(dataset_name="bdanko/sentiment140", train_size=5000, test_size=1000):
    print(f"Loading dataset {dataset_name}...")
    dataset = load_dataset(dataset_name, split="train")
    df = dataset.to_pandas()
    
    # Map label 4 to 1
    df['sentiment'] = df['sentiment'].replace(4, 1)
    
    # Stratified split
    train_df = df.sample(n=train_size, weights=df['sentiment'].map({0: 0.5, 1: 0.5}), random_state=42)
    remaining_df = df.drop(train_df.index)
    
    test_df_neg = remaining_df[remaining_df['sentiment'] == 0].sample(n=test_size // 2, random_state=42)
    test_df_pos = remaining_df[remaining_df['sentiment'] == 1].sample(n=test_size // 2, random_state=42)
    test_df = pd.concat([test_df_neg, test_df_pos]).sample(frac=1, random_state=42)
    
    return DatasetDict({
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True))
    })

dataset = prepare_dataset()
print(dataset)


In [ ]:
model_id = "mistralai/Mistral-7B-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    texts = [f"Sentiment classification. Text: {text}\nSentiment: " for text in examples["text"]]
    tokenized = tokenizer(texts, truncation=True, max_length=128, padding="max_length")
    tokenized["labels"] = examples["sentiment"]
    return tokenized

tokenized_datasets = dataset.map(tokenize_function, batched=True)


In [ ]:
def get_base_model():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model.config.pad_token_id = model.config.eos_token_id
    model = prepare_model_for_kbit_training(model)
    return model

base_model = get_base_model()


In [ ]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=predictions, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=predictions, references=labels)["recall"],
        "f1": f1_metric.compute(predictions=predictions, references=labels)["f1"],
    }


## Baseline, un-tuned


In [ ]:
base_eval_args = TrainingArguments(
    output_dir="./base_eval",
    per_device_eval_batch_size=8,
    bf16=True,
    report_to="none"
)
base_trainer = Trainer(
    model=base_model,
    args=base_eval_args,
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)
base_results = base_trainer.evaluate()
print("Base LLM Zero-Shot Results:")
for k, v in base_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


In [ ]:
def lora_objective(trial):
    r     = trial.suggest_categorical("r", [4, 8, 16, 32])
    alpha = trial.suggest_categorical("alpha", [16, 32, 64])
    lr    = trial.suggest_float("lr", 1e-5, 5e-4, log=True)

    peft_config = LoraConfig(
        r=r, lora_alpha=alpha, target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1, bias="none", task_type="SEQ_CLS"
    )

    # Fresh base model per trial to avoid adapter stacking
    torch.cuda.reset_peak_memory_stats()
    trial_model = get_peft_model(get_base_model(), peft_config)

    training_args = TrainingArguments(
        output_dir="./lora_results",
        per_device_train_batch_size=8,
        num_train_epochs=1,
        weight_decay=0.01,
        learning_rate=lr,
        evaluation_strategy="epoch",
        logging_steps=100,
        bf16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=trial_model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
    )

    trainer.train()
    eval_results = trainer.evaluate()
    trial.set_user_attr("peak_vram_gb", gpu_memory_gb())

    del trainer, trial_model
    torch.cuda.empty_cache()

    return eval_results["eval_f1"]

lora_study = optuna.create_study(direction="maximize")
lora_study.optimize(lora_objective, n_trials=20)


## Adapter Tuning


In [ ]:
def adapter_objective(trial):
    dim     = trial.suggest_categorical("bottleneck_dim", [32, 64, 128]) 
    lr      = trial.suggest_float("lr", 5e-5, 1e-3, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)  # applied as attention dropout on base model

    peft_config = IA3Config(
        target_modules=["k_proj", "v_proj", "down_proj"],
        feedforward_modules=["down_proj"],
        task_type="SEQ_CLS"
    )

    torch.cuda.reset_peak_memory_stats()
    trial_base = get_base_model()

    # attention dropout variation
    for layer in trial_base.model.layers:
        layer.self_attn.dropout = dropout
    trial_model = get_peft_model(trial_base, peft_config)

    training_args = TrainingArguments(
        output_dir="./adapter_results",
        per_device_train_batch_size=8,
        num_train_epochs=1,
        weight_decay=0.01,
        learning_rate=lr,
        evaluation_strategy="epoch",
        logging_steps=100,
        bf16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=trial_model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
    )

    trainer.train()
    eval_results = trainer.evaluate()
    trial.set_user_attr("peak_vram_gb", gpu_memory_gb())

    del trainer, trial_model, trial_base
    torch.cuda.empty_cache()

    return eval_results["eval_f1"]

adapter_study = optuna.create_study(direction="maximize")
adapter_study.optimize(adapter_objective, n_trials=20)


In [ ]:
def train_best(study, is_lora=True):
    best_params = study.best_params
    if is_lora:
        peft_config = LoraConfig(
            r=best_params['r'], lora_alpha=best_params['alpha'],
            target_modules=["q_proj", "v_proj"], lora_dropout=0.1,
            bias="none", task_type="SEQ_CLS"
        )
        lr      = best_params['lr']
        out_dir = "./best_lora"
        hub_id  = "bdanko/mistral-7b-sentiment-lora"
    else:
        peft_config = IA3Config(
            target_modules=["k_proj", "v_proj", "down_proj"],
            feedforward_modules=["down_proj"],
            task_type="SEQ_CLS"
        )
        lr      = best_params['lr']
        out_dir = "./best_adapter"
        hub_id  = "bdanko/mistral-7b-sentiment-adapter"

    torch.cuda.reset_peak_memory_stats()
    base = get_base_model()
    if not is_lora:
        dropout = best_params.get('dropout', 0.0)
        for layer in base.model.layers:
            layer.self_attn.dropout = dropout
    model = get_peft_model(base, peft_config)

    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=8,
        num_train_epochs=3,
        learning_rate=lr,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        bf16=True,
        report_to="none",
        push_to_hub=True,
        hub_model_id=hub_id
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
    )

    trainer.train()
    final_metrics = trainer.evaluate()
    peak_vram = gpu_memory_gb()
    return trainer, final_metrics, peak_vram

print("Training best LoRA model...")
best_lora_trainer, best_lora_metrics, lora_vram = train_best(lora_study, True)
print("Training best Adapter (IA³) model...")
best_adapter_trainer, best_adapter_metrics, adapter_vram = train_best(adapter_study, False)


## Results Export


In [ ]:
lora_trials_df    = lora_study.trials_dataframe()
adapter_trials_df = adapter_study.trials_dataframe()
lora_trials_df["method"]    = "LoRA"
adapter_trials_df["method"] = "Adapter (IA3)"
all_trials = pd.concat([lora_trials_df, adapter_trials_df], ignore_index=True)
all_trials.to_csv("peft_sentiment_optuna_study.csv", index=False)
print("Study saved locally → peft_sentiment_optuna_study.csv")

hf_dataset = Dataset.from_pandas(all_trials)
hf_dataset.push_to_hub("bdanko/peft-sentiment-optuna-study")
print("Study pushed to HF Hub → bdanko/peft-sentiment-optuna-study")

print(f"\nPeak VRAM — LoRA final training : {lora_vram:.3f} GB")
print(f"Peak VRAM — Adapter final training: {adapter_vram:.3f} GB")

lp = lora_study.best_params
ap = adapter_study.best_params

results = pd.DataFrame([
    {
        "Method":    "Base LLM (Zero-Shot)",
        "Best Hyperparams": "N/A",
        "Accuracy":  base_results.get('eval_accuracy', '—'),
        "Precision": base_results.get('eval_precision', '—'),
        "Recall":    base_results.get('eval_recall', '—'),
        "F1":        base_results.get('eval_f1', '—'),
        "Peak VRAM (GB)": '—',
    },
    {
        "Method":    "LoRA (FT)",
        "Best Hyperparams": f"r={lp['r']}, α={lp['alpha']}, lr={lp['lr']:.2e}",
        "Accuracy":  best_lora_metrics.get('eval_accuracy', '—'),
        "Precision": best_lora_metrics.get('eval_precision', '—'),
        "Recall":    best_lora_metrics.get('eval_recall', '—'),
        "F1":        best_lora_metrics.get('eval_f1', '—'),
        "Peak VRAM (GB)": lora_vram,
    },
    {
        "Method":    "Adapter / IA³ (FT)",
        "Best Hyperparams": f"lr={ap['lr']:.2e}, dropout={ap['dropout']:.2f}",
        "Accuracy":  best_adapter_metrics.get('eval_accuracy', '—'),
        "Precision": best_adapter_metrics.get('eval_precision', '—'),
        "Recall":    best_adapter_metrics.get('eval_recall', '—'),
        "F1":        best_adapter_metrics.get('eval_f1', '—'),
        "Peak VRAM (GB)": adapter_vram,
    },
])

print(results.to_string(index=False))


## Qualitative Analysis

In [ ]:
def get_errors(trainer, ds):
    preds       = trainer.predict(ds)
    predictions = np.argmax(preds.predictions, axis=-1)
    labels      = preds.label_ids
    errors      = np.where(predictions != labels)[0]
    return errors, predictions

lora_errors,    lora_preds    = get_errors(best_lora_trainer,    tokenized_datasets["test"])
adapter_errors, adapter_preds = get_errors(best_adapter_trainer, tokenized_datasets["test"])

raw_test = dataset["test"]
label_map = {0: "Negative", 1: "Positive"}

both_wrong = np.intersect1d(lora_errors, adapter_errors)
sample_ids = both_wrong[:3] if len(both_wrong) >= 3 else lora_errors[:3]

print("=" * 70)
print("ERROR ANALYSIS — 3 hard misclassification samples")
print("=" * 70)
for rank, idx in enumerate(sample_ids, 1):
    idx = int(idx)
    true_lbl    = label_map[int(raw_test[idx]['sentiment'])]
    lora_lbl    = label_map[int(lora_preds[idx])]
    adapter_lbl = label_map[int(adapter_preds[idx])]
    print(f"\nSample {rank} (index {idx}):")
    print(f"  Text    : {raw_test[idx]['text']}")
    print(f"  True    : {true_lbl}")
    print(f"  LoRA    : {lora_lbl}  {'✅' if lora_lbl == true_lbl else '❌'}")
    print(f"  IA³     : {adapter_lbl}  {'✅' if adapter_lbl == true_lbl else '❌'}")

print("\n" + "=" * 70)
print("TRAINING STABILITY & VRAM COMPARISON")
print("=" * 70)
lora_vram_trials    = [t.user_attrs.get('peak_vram_gb', float('nan')) for t in lora_study.trials]
adapter_vram_trials = [t.user_attrs.get('peak_vram_gb', float('nan')) for t in adapter_study.trials]
print(f"LoRA    — avg trial VRAM: {np.nanmean(lora_vram_trials):.3f} GB  "
      f"| final training VRAM: {lora_vram:.3f} GB")
print(f"IA³     — avg trial VRAM: {np.nanmean(adapter_vram_trials):.3f} GB  "
      f"| final training VRAM: {adapter_vram:.3f} GB")
lora_f1s    = [t.value for t in lora_study.trials if t.value is not None]
adapter_f1s = [t.value for t in adapter_study.trials if t.value is not None]
print(f"LoRA    — F1 std across 20 trials: {np.std(lora_f1s):.4f}  (lower = more stable)")
print(f"IA³     — F1 std across 20 trials: {np.std(adapter_f1s):.4f}")
